In [2]:
import sys
import os

!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

sys.path.append('/content/proyecto-transformacion-texto-imagen')

print("✅ Entorno configurado correctamente.")

Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 478, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 478 (delta 79), reused 113 (delta 31), pack-reused 296 (from 1)
Receiving objects: 100% (478/478), 15.38 MiB | 15.51 MiB/s, done.
Resolving deltas: 100% (229/229), done.
✅ Entorno configurado correctamente.


In [3]:
import sys, os, gc, json
import numpy as np
import polars as pl
import torch
import joblib
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset
from scipy import stats
from skimage.feature import local_binary_pattern
from skimage.filters import sobel
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
PERMUTATION_SEED = 99   # Seed distinto al del pipeline original (42)
                        # para que la permutación sea reproducible pero no idéntica al split
set_seed(42)            # Seed de PyTorch/HuggingFace igual al original
print("✅ Setup completo")

✅ Setup completo


In [4]:
PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
ruta = f'{PROJECT_ROOT}/data/processed'

train_df = pl.read_csv(f'{ruta}/train.csv')
val_df   = pl.read_csv(f'{ruta}/validation.csv')
test_df  = pl.read_csv(f'{ruta}/test.csv')

X_train = train_df.get_column('text').to_list()
X_val   = val_df.get_column('text').to_list()
X_test  = test_df.get_column('text').to_list()

# Etiquetas REALES (para val y test — nunca se permutan)
y_train_real = train_df.get_column('manual_classification').to_numpy()
y_val        = val_df.get_column('manual_classification').to_numpy()
y_test       = test_df.get_column('manual_classification').to_numpy()

# Etiquetas PERMUTADAS para train
rng = np.random.default_rng(seed=PERMUTATION_SEED)
y_train_perm = rng.permutation(y_train_real)

# Verificación de sanidad: la distribución de clases debe ser idéntica
# (permutamos orden, no composición)
assert y_train_perm.sum() == y_train_real.sum(), \
    "ERROR: la permutación cambió el balance de clases"

print(f"✓ Train original  — Dep: {y_train_real.sum()} | No-dep: {(y_train_real==0).sum()}")
print(f"✓ Train permutado — Dep: {y_train_perm.sum()} | No-dep: {(y_train_perm==0).sum()}")
print(f"✓ Primeras 10 etiquetas reales:    {y_train_real[:10]}")
print(f"✓ Primeras 10 etiquetas permutadas: {y_train_perm[:10]}")

✓ Train original  — Dep: 353 | No-dep: 555
✓ Train permutado — Dep: 353 | No-dep: 555
✓ Primeras 10 etiquetas reales:    [0 0 0 0 0 0 0 0 0 0]
✓ Primeras 10 etiquetas permutadas: [1 0 0 0 0 0 1 0 0 0]


In [5]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision_macro': p, 'recall_macro': r, 'f1_macro': f1}

print("✅ Clases auxiliares definidas")

✅ Clases auxiliares definidas


In [6]:
PERM_CKPT_DIR = f'{PROJECT_ROOT}/models/checkpoints/perm_control'
os.makedirs(PERM_CKPT_DIR, exist_ok=True)

# Hiperparámetros IDÉNTICOS a Exp 4.1
HP = {
    'learning_rate':  1.0643090454382045e-05,
    'batch_size':     8,
    'num_epochs':     4,
    'weight_decay':   0.02804067960331229,
    'warmup_ratio':   0.11034302016226369,
    'max_length':     256
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_perm = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Dataset con etiquetas PERMUTADAS en train, REALES en val
train_ds_perm = DepressionDataset(X_train, y_train_perm, tokenizer, HP['max_length'])
val_ds        = DepressionDataset(X_val,   y_val,         tokenizer, HP['max_length'])

training_args = TrainingArguments(
    output_dir                  = PERM_CKPT_DIR,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_strategy            = 'epoch',
    learning_rate               = HP['learning_rate'],
    per_device_train_batch_size = HP['batch_size'],
    per_device_eval_batch_size  = HP['batch_size'],
    num_train_epochs            = HP['num_epochs'],
    weight_decay                = HP['weight_decay'],
    warmup_ratio                = HP['warmup_ratio'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    save_total_limit            = 1,
    fp16                        = True,
    seed                        = 42,
    report_to                   = 'none',
)

trainer_perm = Trainer(
    model           = model_perm,
    args            = training_args,
    train_dataset   = train_ds_perm,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("... Entrenando BETO con etiquetas permutadas...")
print(f"   Hiperparámetros: {HP}")
trainer_perm.train()

# Evaluar en val con etiquetas reales — esperamos rendimiento ~azar (~0.50 F1-Macro)
metrics_val = trainer_perm.evaluate()
print(f"\n... F1-Val con etiquetas permutadas: {metrics_val['eval_f1_macro']:.4f}")
print(f"   (Esperado: cercano a 0.50 — rendimiento de azar)")

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.493797,0.777496,4,0.561404,0.424020,0.469805,0.408223



... F1-Val con etiquetas permutadas: 0.4082
   (Esperado: cercano a 0.50 — rendimiento de azar)


In [7]:
from transformers import BertForSequenceClassification
import gc

def extraer_embeddings(texts, model_or_path, tokenizer, batch_size=16, max_length=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if isinstance(model_or_path, str):
        model_cls = BertForSequenceClassification.from_pretrained(
            model_or_path, ignore_mismatched_sizes=True).to(device)
    else:
        model_cls = model_or_path.to(device)

    encoder = model_cls.bert
    encoder.eval()
    all_emb = []

    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out = encoder(**inputs)
            all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())

    del model_cls, encoder
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(all_emb)

print("--> Extrayendo embeddings del modelo permutado...")
emb_train_perm = extraer_embeddings(X_train, trainer_perm.model, tokenizer)
emb_val_perm   = extraer_embeddings(X_val,   trainer_perm.model, tokenizer)
emb_test_perm  = extraer_embeddings(X_test,  trainer_perm.model, tokenizer)

print(f". . .Embeddings extraídos — Train: {emb_train_perm.shape} | Val: {emb_val_perm.shape} | Test: {emb_test_perm.shape}")

# Liberar memoria del modelo permutado — ya no lo necesitas
del trainer_perm, model_perm
torch.cuda.empty_cache()
gc.collect()

--> Extrayendo embeddings del modelo permutado...
. . .Embeddings extraídos — Train: (908, 768) | Val: (114, 768) | Test: (114, 768)


4487

In [13]:
# ── Fallback si el warm-up no funciona ────────────────────────────────────
# Usamos los componentes ya transformados del pkl como referencia para el
# scaler, y re-fiteamos UMAP SOLO sobre emb_train_perm con los mismos
# hiperparámetros del contrato técnico.
# IMPORTANTE: esto produce una proyección UMAP distinta a la original —
# se documenta como limitación del control, no invalida el experimento.

import umap

print("⚠️  Usando fallback: re-fit UMAP con hiperparámetros del contrato técnico")
reducer_perm = umap.UMAP(
    n_components  = 5,
    metric        = 'cosine',
    n_neighbors   = 15,
    min_dist      = 0.1,
    random_state  = 42      # mismo seed que el original
)

umap_train_perm = reducer_perm.fit_transform(emb_train_perm)
umap_val_perm   = reducer_perm.transform(emb_val_perm)
umap_test_perm  = reducer_perm.transform(emb_test_perm)

# Fittear scaler SOLO sobre train permutado (sin leakage)
scaler_perm = MinMaxScaler()
umap_train_perm = np.clip(scaler_perm.fit_transform(umap_train_perm), 0.0, 1.0)
umap_val_perm   = np.clip(scaler_perm.transform(umap_val_perm),       0.0, 1.0)
umap_test_perm  = np.clip(scaler_perm.transform(umap_test_perm),      0.0, 1.0)

print(f"✅ Fallback completado")
print(f"   Train: {umap_train_perm.shape} | rango [{umap_train_perm.min():.4f}, {umap_train_perm.max():.4f}]")

⚠️  Usando fallback: re-fit UMAP con hiperparámetros del contrato técnico
✅ Fallback completado
   Train: (908, 5) | rango [0.0000, 1.0000]


In [14]:
# Opción A: calcular features directamente sobre componentes UMAP
# (válido como control de separabilidad en el espacio reducido)

def calcular_features_umap(umap_components, labels):
    """
    Calcula estadísticas básicas de separabilidad sobre los 5 componentes UMAP.
    No requiere generar imágenes — es un control en el espacio de parámetros.
    """
    resultados = {}
    for i in range(umap_components.shape[1]):
        comp = umap_components[:, i]
        dep    = comp[labels == 1]
        nodep  = comp[labels == 0]
        t_stat, p_val = stats.ttest_ind(dep, nodep, equal_var=False)  # Welch
        resultados[f'umap_{i}'] = {
            'mean_dep':   dep.mean(),
            'mean_nodep': nodep.mean(),
            'std_dep':    dep.std(),
            'std_nodep':  nodep.std(),
            't_stat':     t_stat,
            'p_value':    p_val
        }
    return resultados

# Análisis sobre el conjunto completo (train+val+test) con etiquetas REALES
# para evaluar si los embeddings permutados mantienen separabilidad
umap_all_perm   = np.vstack([umap_train_perm, umap_val_perm, umap_test_perm])
y_all_real      = np.concatenate([y_train_real, y_val, y_test])

features_perm = calcular_features_umap(umap_all_perm, y_all_real)

print("\n📊 Separabilidad inter-clase en componentes UMAP — modelo permutado")
print(f"{'Comp':<8} {'Mean Dep':>10} {'Mean NoDep':>10} {'t-stat':>8} {'p-value':>10} {'Sig':>5}")
print("-" * 55)
n_sig = 0
for comp, vals in features_perm.items():
    sig = "✓" if vals['p_value'] < 0.05 else "—"
    if vals['p_value'] < 0.05:
        n_sig += 1
    print(f"{comp:<8} {vals['mean_dep']:>10.4f} {vals['mean_nodep']:>10.4f} "
          f"{vals['t_stat']:>8.3f} {vals['p_value']:>10.4f} {sig:>5}")

print(f"\n✅ Componentes significativos: {n_sig}/5")
print(f"   (Esperado si control es válido: 0 o 1 — sin separabilidad sistemática)")


📊 Separabilidad inter-clase en componentes UMAP — modelo permutado
Comp       Mean Dep Mean NoDep   t-stat    p-value   Sig
-------------------------------------------------------
umap_0       0.4366     0.3867    3.255     0.0012     ✓
umap_1       0.5497     0.5250    2.109     0.0351     ✓
umap_2       0.5103     0.5526   -3.119     0.0019     ✓
umap_3       0.4388     0.5178   -5.807     0.0000     ✓
umap_4       0.5452     0.6112   -4.416     0.0000     ✓

✅ Componentes significativos: 5/5
   (Esperado si control es válido: 0 o 1 — sin separabilidad sistemática)


In [18]:
# ── Celda adicional: comparación cuantitativa permutado vs. original ──────
# Cargar el UMAP y scaler compartidos — fiteados sobre X_train real de Exp 4.1
pkg_umap = joblib.load(f'{PROJECT_ROOT}/data/shared/umap_params.pkl')


# Componentes UMAP originales (del pkl)
umap_all_orig = np.vstack([
    pkg_umap['train']['params'],
    pkg_umap['val']['params'],
    pkg_umap['test']['params']
])
y_all_real = np.concatenate([y_train_real, y_val, y_test])

# Recalcular features del pipeline original con la misma función
features_orig = calcular_features_umap(umap_all_orig, y_all_real)

# Tabla comparativa: t-stats y Cohen's d para ambos modelos
print(f"\n{'='*72}")
print(f"{'Comparación: Pipeline Original vs. Control de Permutación':^72}")
print(f"{'='*72}")
print(f"{'Comp':<8} {'|t| Orig':>10} {'|t| Perm':>10} {'Ratio':>8} {'p Orig':>10} {'p Perm':>10}")
print(f"{'-'*72}")

for comp in features_orig:
    t_orig = abs(features_orig[comp]['t_stat'])
    t_perm = abs(features_perm[comp]['t_stat'])
    ratio  = t_orig / t_perm if t_perm > 0 else float('inf')
    p_orig = features_orig[comp]['p_value']
    p_perm = features_perm[comp]['p_value']
    print(f"{comp:<8} {t_orig:>10.3f} {t_perm:>10.3f} {ratio:>8.2f}x {p_orig:>10.4f} {p_perm:>10.4f}")

# Cohen's d para cuantificar el tamaño de efecto en cada modelo
def cohen_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    s_pooled = np.sqrt((group1.std()**2 + group2.std()**2) / 2)
    return abs(group1.mean() - group2.mean()) / s_pooled if s_pooled > 0 else 0

print(f"\n{'='*55}")
print(f"{'Cohens d — tamaño de efecto':^55}")
print(f"{'='*55}")
print(f"{'Comp':<8} {'d Orig':>10} {'d Perm':>10} {'Reducción':>12}")
print(f"{'-'*55}")

for comp in features_orig:
    vals_o = features_orig[comp]
    vals_p = features_perm[comp]

    dep_orig   = umap_all_orig[y_all_real == 1, int(comp[-1])]
    nodep_orig = umap_all_orig[y_all_real == 0, int(comp[-1])]
    dep_perm   = umap_all_perm[y_all_real == 1, int(comp[-1])]
    nodep_perm = umap_all_perm[y_all_real == 0, int(comp[-1])]

    d_orig = cohen_d(dep_orig, nodep_orig)
    d_perm = cohen_d(dep_perm, nodep_perm)
    reduccion = (1 - d_perm / d_orig) * 100 if d_orig > 0 else 0

    print(f"{comp:<8} {d_orig:>10.4f} {d_perm:>10.4f} {reduccion:>11.1f}%")


       Comparación: Pipeline Original vs. Control de Permutación        
Comp       |t| Orig   |t| Perm    Ratio     p Orig     p Perm
------------------------------------------------------------------------
umap_0       31.501      3.255     9.68x     0.0000     0.0012
umap_1        8.314      2.109     3.94x     0.0000     0.0351
umap_2       23.674      3.119     7.59x     0.0000     0.0019
umap_3       21.771      5.807     3.75x     0.0000     0.0000
umap_4       32.800      4.416     7.43x     0.0000     0.0000

              Cohens d — tamaño de efecto              
Comp         d Orig     d Perm    Reducción
-------------------------------------------------------
umap_0       1.9034     0.1970        89.6%
umap_1       0.5064     0.1265        75.0%
umap_2       1.4284     0.1884        86.8%
umap_3       1.3129     0.3535        73.1%
umap_4       1.9925     0.2673        86.6%
